# CIS480 Historical Stock Market Analysis

## Five Semiconductor Stocks vs. SPY

**Historical Period:** January 1, 2021 – December 31, 2025  
**Stocks:** AMD, NVIDIA (NVDA), Intel (INTC), Taiwan Semiconductor Manufacturing (TSM), Broadcom (AVGO)  
**Benchmark:** SPDR S&P 500 ETF Trust (SPY)  
**Data Source:** Yahoo Finance through the yfinance Python library  
**Frequency:** Daily  
**Price Adjustment:** Auto-adjusted prices  

### Research Question
How did the returns, volatility, and drawdowns of AMD, NVIDIA, Intel, TSM, and Broadcom compare with the SPY benchmark from 2021 through 2025?


In [ ]:
%pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

print("yfinance version:", yf.__version__)
print("pandas version:", pd.__version__)

In [ ]:
# CIS480 Project Parameters

tickers = ["AMD", "NVDA", "INTC", "TSM", "AVGO", "SPY"]

start_date = "2021-01-01"
end_date = "2026-01-01"  # yfinance end date is exclusive

interval = "1d"
auto_adjust = True

print("Tickers:", tickers)
print("Historical Period: 2021-01-01 through 2025-12-31")
print("Interval:", interval)
print("Auto Adjust:", auto_adjust)


In [ ]:
data = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    interval=interval,
    auto_adjust=auto_adjust,
    group_by="column",
    threads=False,
    progress=True
)

data.head()


In [ ]:
# Export the untouched yfinance dataset
data.to_csv(
    "outputs/CIS480_Raw_Stock_Data_2021_2025.csv"
)

print("Raw dataset successfully saved.")

In [ ]:
print("Dataset shape:", data.shape)
print()
print("Date range:")
print("First date:", data.index.min())
print("Last date:", data.index.max())
print()
print("Missing Close values:")
print(data["Close"].isna().sum())

In [ ]:
data.tail()

## Data Retrieval Information

- **Data Source:** Yahoo Finance using the yfinance Python library
- **Stocks:** AMD (AMD), NVIDIA (NVDA), Intel (INTC), Taiwan Semiconductor Manufacturing (TSM), Broadcom (AVGO)
- **Benchmark:** SPDR S&P 500 ETF Trust (SPY)
- **Historical Period:** January 1, 2021 through December 31, 2025
- **Interval:** Daily
- **Price Adjustment:** Auto-adjusted (`auto_adjust=True`)
- **Retrieval Date:** Record the date when this notebook is run for the A04 evidence package.


## A04 First Working Vertical Slice

This section documents a small end-to-end path from representative historical-price input to a labeled analytical output. It validates the required data, calculates a defined return measure, and reports a comparison that can be checked before the full analysis continues.

**Expected output:** All six requested securities are present, the date range falls within 2021–2025, and a labeled cumulative-return comparison is produced.


In [ ]:
# Validate the downloaded input before analysis
required_price_field = "Close"
retrieved_tickers = sorted(data[required_price_field].columns.tolist())
missing_tickers = sorted(set(tickers) - set(retrieved_tickers))

print("Required price field present:", required_price_field in data.columns.get_level_values(0))
print("Requested tickers:", sorted(tickers))
print("Retrieved tickers:", retrieved_tickers)
print("Missing requested tickers:", missing_tickers)
print("First trading date:", data.index.min())
print("Last trading date:", data.index.max())
print("Duplicate dates:", int(data.index.duplicated().sum()))
print("Missing Close values by ticker:")
print(data[required_price_field].isna().sum())

validation_passed = (
    required_price_field in data.columns.get_level_values(0)
    and len(missing_tickers) == 0
    and data.index.duplicated().sum() == 0
)
print("\nA04 input validation status:", "PASSED" if validation_passed else "PARTIALLY PASSED / REVIEW NEEDED")


In [ ]:
# First-build calculation: cumulative return from first to last available Close
first_build = pd.DataFrame({
    "First_Close": data["Close"].apply(lambda s: s.dropna().iloc[0]),
    "Last_Close": data["Close"].apply(lambda s: s.dropna().iloc[-1])
})
first_build["Cumulative_Return_Pct"] = (
    (first_build["Last_Close"] / first_build["First_Close"] - 1) * 100
).round(2)
first_build.index.name = "Ticker"
first_build = first_build.reset_index()
first_build["Company"] = first_build["Ticker"].map({
    "AMD": "AMD", "NVDA": "NVIDIA", "INTC": "Intel",
    "TSM": "TSMC", "AVGO": "Broadcom", "SPY": "SPY (S&P 500 Benchmark)"
})
first_build[["Ticker", "Company", "Cumulative_Return_Pct"]].sort_values("Ticker")


### First-Build Test Record

- **Representative input:** Daily auto-adjusted historical prices downloaded for the six selected securities.
- **Validation:** Required Close field, ticker coverage, date range, duplicate dates, and missing Close values are reported above.
- **Actual output:** The executed cells above provide the actual validation status and labeled cumulative-return comparison.
- **Test status:** Record as **Passed**, **Partially Passed**, or **Failed** after running the notebook and reviewing the displayed output.
- **Known limitation:** Historical returns, volatility, and drawdowns describe the selected historical period and do not guarantee future investment performance. Trading calendars and missing observations can also differ among securities, including TSM.
- **Verification:** Before submission, manually verify at least one cumulative-return result using `(last adjusted Close / first adjusted Close - 1) × 100`.


## Final Team ETL and Analytical Dataset

The following section consolidates the team's final ETL and KPI workflow into one standardized long-format dataset for Power BI and Python analysis.

The original A04 retrieval and first working vertical slice above are preserved. The final workflow below incorporates the finalized ETL structure and supporting measures developed by Rey while retaining the project's validated six-security scope:

- AMD
- NVIDIA (NVDA)
- Intel (INTC)
- Taiwan Semiconductor Manufacturing (TSM)
- Broadcom (AVGO)
- SPY benchmark

All analytical percentage fields remain numeric proportions in the primary dataset so they can be formatted correctly in Power BI.


In [ ]:
# Reshape the already-downloaded yfinance data into the final long-format schema

company_names = {
    "AMD": "AMD",
    "NVDA": "NVIDIA",
    "INTC": "Intel",
    "TSM": "TSMC",
    "AVGO": "Broadcom",
    "SPY": "SPY (S&P 500 Benchmark)"
}

processed_frames = []

for ticker in tickers:
    df_ticker = pd.DataFrame({
        "Trading_Date": data.index,
        "Adj_Close": data["Close"][ticker].values,
        "Volume": data["Volume"][ticker].values
    }).dropna(subset=["Adj_Close"]).reset_index(drop=True)

    # Primary return and risk fields
    df_ticker["Daily_Return"] = df_ticker["Adj_Close"].pct_change()

    start_price = df_ticker["Adj_Close"].iloc[0]
    df_ticker["Cumulative_Return"] = (
        df_ticker["Adj_Close"] / start_price
    ) - 1

    df_ticker["Rolling_Peak"] = df_ticker["Adj_Close"].cummax()
    df_ticker["Drawdown"] = (
        df_ticker["Adj_Close"] - df_ticker["Rolling_Peak"]
    ) / df_ticker["Rolling_Peak"]

    # Metadata
    df_ticker["Ticker"] = ticker
    df_ticker["Company"] = company_names[ticker]
    df_ticker["Asset_Class"] = (
        "Benchmark" if ticker == "SPY" else "Equity"
    )

    processed_frames.append(df_ticker)

clean_long_df = pd.concat(processed_frames, ignore_index=True)

clean_long_df["Trading_Date"] = pd.to_datetime(
    clean_long_df["Trading_Date"]
)

clean_long_df.sort_values(
    by=["Ticker", "Trading_Date"],
    inplace=True
)

clean_long_df.reset_index(drop=True, inplace=True)

print("Final long-format dataset shape:", clean_long_df.shape)
print("First trading date:", clean_long_df["Trading_Date"].min())
print("Last trading date:", clean_long_df["Trading_Date"].max())
print("Tickers:", sorted(clean_long_df["Ticker"].unique()))


## Supporting Measures

The final analytical dataset includes four supporting measures in addition to the three primary KPIs:

1. **Trading Day Count** — sequential trading-day index for each ticker.
2. **Normalized Price** — each security indexed to 100 on its first trading date.
3. **Benchmark Difference** — each security's daily return minus SPY's daily return.
4. **Annual Return** — return from the first available trading day to the last available trading day within each calendar year.


In [ ]:
# Trading-Day Count
clean_long_df["Trading_Day_Count"] = (
    clean_long_df.groupby("Ticker").cumcount() + 1
)

# Normalized Price: base value of 100 on the first trading date
p0_series = clean_long_df.groupby("Ticker")["Adj_Close"].transform("first")
clean_long_df["Normalized_Price"] = (
    clean_long_df["Adj_Close"] / p0_series
) * 100

# Benchmark Difference: ticker daily return minus SPY daily return
spy_returns = (
    clean_long_df[clean_long_df["Ticker"] == "SPY"]
    .set_index("Trading_Date")["Daily_Return"]
)

clean_long_df["SPY_Daily_Return"] = (
    clean_long_df["Trading_Date"].map(spy_returns)
)

clean_long_df["Benchmark_Difference"] = (
    clean_long_df["Daily_Return"]
    - clean_long_df["SPY_Daily_Return"]
)

# SPY compared with itself should be 0 where a daily return exists
clean_long_df.drop(columns=["SPY_Daily_Return"], inplace=True)

# Annual Return
clean_long_df["Year"] = clean_long_df["Trading_Date"].dt.year

clean_long_df["Annual_Return"] = (
    clean_long_df
    .groupby(["Ticker", "Year"])["Adj_Close"]
    .transform(lambda x: (x.iloc[-1] / x.iloc[0]) - 1)
)

# Final chronological ordering
clean_long_df.sort_values(
    by=["Trading_Date", "Ticker"],
    inplace=True
)

clean_long_df.reset_index(drop=True, inplace=True)

clean_long_df.head(12)


## Week 5 Data Source Quality and Preprocessing Validation

This section profiles the final analytical dataset before export. The purpose is to verify the dataset schema, row count, ticker coverage, date coverage, key uniqueness, missingness, and reasonable value ranges.

Missing values are reviewed based on their meaning rather than automatically being replaced with zero. The first Daily_Return value for each security is expected to be missing because there is no previous trading day in the dataset for calculating a percentage change. Benchmark_Difference is also expected to be missing on the first observation for each security because it depends on Daily_Return.

The checks below validate the structure of the dataset used for the final analysis and Power BI dashboard.


In [ ]:
# ============================================================
# WEEK 5 DATA QUALITY PROFILE AND VALIDATION
# ============================================================

print("=" * 65)
print("CIS480 WEEK 5 DATA QUALITY PROFILE")
print("=" * 65)

# 1. EXPECTED SCHEMA
expected_columns = [
    "Trading_Date", "Adj_Close", "Volume", "Daily_Return",
    "Cumulative_Return", "Rolling_Peak", "Drawdown", "Ticker",
    "Company", "Asset_Class", "Trading_Day_Count",
    "Normalized_Price", "Benchmark_Difference", "Year",
    "Annual_Return"
]

actual_columns = clean_long_df.columns.tolist()
schema_passed = set(expected_columns) == set(actual_columns)

print("\n1. SCHEMA CHECK")
print("Expected columns:", len(expected_columns))
print("Actual columns:", len(actual_columns))
print("Status:", "PASSED" if schema_passed else "REVIEW NEEDED")

if not schema_passed:
    print("Missing columns:", sorted(set(expected_columns) - set(actual_columns)))
    print("Unexpected columns:", sorted(set(actual_columns) - set(expected_columns)))

# 2. ROW COUNT AND COVERAGE
row_count = len(clean_long_df)
trading_dates = clean_long_df["Trading_Date"].nunique()
security_count = clean_long_df["Ticker"].nunique()

print("\n2. ROW COUNT AND COVERAGE")
print("Total rows:", row_count)
print("Unique trading dates:", trading_dates)
print("Unique securities:", security_count)
print("First trading date:", clean_long_df["Trading_Date"].min())
print("Last trading date:", clean_long_df["Trading_Date"].max())

# 3. REQUIRED TICKER CHECK
expected_tickers_w05 = {"AMD", "AVGO", "INTC", "NVDA", "SPY", "TSM"}
actual_tickers_w05 = set(clean_long_df["Ticker"].unique())
ticker_check = actual_tickers_w05 == expected_tickers_w05

print("\n3. REQUIRED SECURITY CHECK")
print("Expected:", sorted(expected_tickers_w05))
print("Actual:", sorted(actual_tickers_w05))
print("Status:", "PASSED" if ticker_check else "REVIEW NEEDED")

# 4. KEY UNIQUENESS
duplicate_keys = clean_long_df.duplicated(
    subset=["Trading_Date", "Ticker"]
).sum()

print("\n4. TICKER + DATE KEY UNIQUENESS")
print("Duplicate ticker/date rows:", duplicate_keys)
print("Status:", "PASSED" if duplicate_keys == 0 else "REVIEW NEEDED")

# 5. MISSINGNESS PROFILE
missing_profile = pd.DataFrame({
    "Missing_Count": clean_long_df.isna().sum(),
    "Missing_Percent": (clean_long_df.isna().mean() * 100).round(2)
})

print("\n5. MISSINGNESS PROFILE")
display(missing_profile)

# Expected structural missingness: one first-day return per security.
expected_daily_return_missing = security_count
actual_daily_return_missing = clean_long_df["Daily_Return"].isna().sum()
actual_benchmark_difference_missing = clean_long_df[
    "Benchmark_Difference"
].isna().sum()

print("Expected Daily_Return missing values:", expected_daily_return_missing)
print("Actual Daily_Return missing values:", actual_daily_return_missing)
print(
    "Actual Benchmark_Difference missing values:",
    actual_benchmark_difference_missing
)

daily_return_missing_passed = (
    actual_daily_return_missing == expected_daily_return_missing
)
benchmark_missing_passed = (
    actual_benchmark_difference_missing == expected_daily_return_missing
)

print(
    "Daily_Return missingness:",
    "PASSED" if daily_return_missing_passed else "REVIEW NEEDED"
)
print(
    "Benchmark_Difference missingness:",
    "PASSED" if benchmark_missing_passed else "REVIEW NEEDED"
)

# 6. BASIC RANGE AND VALIDITY CHECKS
positive_prices = (clean_long_df["Adj_Close"] > 0).all()
nonnegative_volume = (clean_long_df["Volume"] >= 0).all()
valid_daily_returns = (clean_long_df["Daily_Return"].dropna() > -1).all()
valid_drawdown = (clean_long_df["Drawdown"].dropna() <= 0).all()
positive_normalized_price = (clean_long_df["Normalized_Price"] > 0).all()
valid_trading_day_count = (clean_long_df["Trading_Day_Count"] >= 1).all()
valid_years = clean_long_df["Year"].between(2021, 2025).all()

range_checks = pd.DataFrame({
    "Check": [
        "Adjusted Close > 0",
        "Volume >= 0",
        "Daily Return > -100%",
        "Drawdown <= 0",
        "Normalized Price > 0",
        "Trading Day Count >= 1",
        "Year between 2021 and 2025"
    ],
    "Passed": [
        positive_prices, nonnegative_volume, valid_daily_returns,
        valid_drawdown, positive_normalized_price,
        valid_trading_day_count, valid_years
    ]
})

range_checks["Status"] = range_checks["Passed"].map(
    {True: "PASSED", False: "REVIEW NEEDED"}
)

print("\n6. BASIC RANGE AND VALIDITY CHECKS")
display(range_checks[["Check", "Status"]])

# 7. OBSERVED NUMERIC RANGES
range_profile = clean_long_df[
    [
        "Adj_Close", "Volume", "Daily_Return", "Cumulative_Return",
        "Drawdown", "Normalized_Price", "Benchmark_Difference",
        "Annual_Return"
    ]
].agg(["min", "max"]).T

print("\n7. OBSERVED NUMERIC RANGES")
display(range_profile)

# 8. RECORD COUNTS BY SECURITY
records_by_ticker = (
    clean_long_df.groupby("Ticker")
    .size()
    .reset_index(name="Row_Count")
)

print("\n8. RECORD COUNTS BY SECURITY")
display(records_by_ticker)

# 9. FINAL WEEK 5 VALIDATION STATUS
week5_checks = [
    schema_passed,
    ticker_check,
    duplicate_keys == 0,
    clean_long_df["Adj_Close"].notna().all(),
    daily_return_missing_passed,
    benchmark_missing_passed,
    positive_prices,
    nonnegative_volume,
    valid_daily_returns,
    valid_drawdown,
    positive_normalized_price,
    valid_trading_day_count,
    valid_years
]

week5_validation_passed = all(week5_checks)

print("\n" + "=" * 65)
print(
    "WEEK 5 DATA QUALITY STATUS:",
    "PASSED" if week5_validation_passed else "REVIEW NEEDED"
)
print("=" * 65)


## Primary KPI Summary

The three primary project KPIs are:

- **Cumulative Return** — total price performance from the first to the last available trading date.
- **Annualized Volatility** — standard deviation of daily returns multiplied by the square root of 252 trading days.
- **Maximum Drawdown** — the largest decline from a running peak during the analysis period.


In [ ]:
kpi_summary = (
    clean_long_df
    .groupby("Ticker")
    .agg(
        Asset_Class=("Asset_Class", "first"),
        Start_Price=("Adj_Close", "first"),
        End_Price=("Adj_Close", "last"),
        Cumulative_Return=("Cumulative_Return", "last"),
        Annualized_Volatility=(
            "Daily_Return",
            lambda x: x.std() * np.sqrt(252)
        ),
        Max_Drawdown=("Drawdown", "min")
    )
    .reset_index()
)

kpi_summary


In [ ]:
# Display a presentation-friendly KPI table while keeping the source table numeric

kpi_summary_display = kpi_summary.copy()

for col in [
    "Cumulative_Return",
    "Annualized_Volatility",
    "Max_Drawdown"
]:
    kpi_summary_display[col] = (
        kpi_summary_display[col] * 100
    ).round(2).astype(str) + "%"

kpi_summary_display


## Validation Checks

These checks confirm that the final master dataset contains the expected securities, date coverage, unique ticker/date observations, and internally consistent KPI calculations before export.


In [ ]:
expected_tickers = set(tickers)
actual_tickers = set(clean_long_df["Ticker"].unique())

assert actual_tickers == expected_tickers, (
    f"Ticker mismatch. Expected {expected_tickers}, got {actual_tickers}"
)

assert not clean_long_df.duplicated(
    subset=["Trading_Date", "Ticker"]
).any(), "Duplicate ticker/date rows found."

assert clean_long_df["Adj_Close"].notna().all(), (
    "Missing adjusted close values found."
)

# Recalculate final cumulative returns from first/last prices
validation_returns = (
    clean_long_df.groupby("Ticker")["Adj_Close"]
    .agg(["first", "last"])
)

validation_returns["Recalculated_Return"] = (
    validation_returns["last"]
    / validation_returns["first"]
) - 1

reported_returns = (
    kpi_summary
    .set_index("Ticker")["Cumulative_Return"]
)

assert np.allclose(
    validation_returns["Recalculated_Return"].sort_index(),
    reported_returns.sort_index(),
    rtol=1e-10,
    atol=1e-12
), "Cumulative return validation failed."

print("All final validation checks passed.")
print("Rows:", len(clean_long_df))
print("Trading dates:", clean_long_df["Trading_Date"].nunique())
print("Securities:", clean_long_df["Ticker"].nunique())


## Final Exports for Power BI and Project Evidence

The master notebook produces:

- `cleaned_market_data_2021_2025.csv` — primary numeric dataset for Power BI and Python.
- `cleaned_market_data_formatted_2021_2025.csv` — human-readable percentage-formatted copy.
- `kpi_summary_metrics_2021_2025.csv` — numeric summary of the three primary KPIs.

The numeric files should be used for calculations and Power BI modeling. Percentage formatting should be applied in Power BI rather than converting analytical fields to text.


In [ ]:
# Primary numeric dataset
numeric_columns = [
    "Trading_Date",
    "Ticker",
    "Company",
    "Asset_Class",
    "Adj_Close",
    "Volume",
    "Daily_Return",
    "Cumulative_Return",
    "Drawdown",
    "Trading_Day_Count",
    "Normalized_Price",
    "Benchmark_Difference",
    "Year",
    "Annual_Return"
]

final_numeric_df = clean_long_df[numeric_columns].copy()

final_numeric_df.to_csv(
    "outputs/cleaned_market_data_2021_2025.csv",
    index=False
)

# Human-readable formatted copy
formatted_df = final_numeric_df.copy()

pct_columns = [
    "Daily_Return",
    "Cumulative_Return",
    "Drawdown",
    "Benchmark_Difference",
    "Annual_Return"
]

for col in pct_columns:
    formatted_df[col] = (
        formatted_df[col] * 100
    ).round(2).astype(str) + "%"

formatted_df["Normalized_Price"] = (
    formatted_df["Normalized_Price"].round(2)
)

formatted_df.to_csv(
    "outputs/cleaned_market_data_formatted_2021_2025.csv",
    index=False
)

# Numeric KPI summary
kpi_summary.to_csv(
    "outputs/kpi_summary_metrics_2021_2025.csv",
    index=False
)

print("Final project files exported successfully.")


## Team Validation Note

This master notebook consolidates the original A04 workflow with the finalized ETL and supporting-measure work. The three primary KPI outputs were also independently reviewed in the team's separate KPI visualization/validation notebook.

The individual source notebooks should remain preserved as evidence of each team member's contribution. This master notebook is the consolidated team analysis file for the final project workflow.
